# 02 - Data preprocessing and leakage-aware splits

This project trains only on the 15 configured ISL classes. The other five source-dataset folders are ignored by indexing and never enter splits or model outputs. Rerun after changing the data or split configuration.

The configured signer split keeps each inferred signer and duplicate group in only one partition. Signer IDs are inferred from the dataset User folders.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, project_path, resolve_path
from src.dataset import create_splits, discover_images, load_split_manifest, save_split_manifest
from src.transforms import build_transforms
from src.utils import set_seed

config = load_config()
seed = int(config["project"].get("seed", 42))
set_seed(seed)
root = resolve_path(config, config["data"].get("dataset_root"))
manifest_path = project_path(config, "splits") / "split_manifest.json"

if root is None:
    print("Dataset path is not configured. Set data.dataset_root in configs/config.yaml to the extracted folder containing class directories.")
    manifest = None
elif not root.is_dir():
    print(f"Dataset directory does not exist: {root}. Correct data.dataset_root before proceeding.")
    manifest = None
else:
    records, class_to_idx = discover_images(config)
    ratios = config["data"]["split"]
    splits, split_mode = create_splits(records, class_to_idx, ratios, seed, strategy=config['data'].get('split_strategy', 'auto'))
    saved_manifest = save_split_manifest(splits, class_to_idx, split_mode, seed, config)
    manifest = load_split_manifest(saved_manifest)
    print(f"Classes: {len(class_to_idx)} | readable images: {len(records)}")
    print(f"Split strategy: {split_mode}")
    print(f"Saved manifest: {saved_manifest}")

## Validate class mapping and split integrity

The manifest contains only the 15 configured class folders. Checks ensure every selected class appears in each partition and no duplicate group is shared between partitions. No test-set metrics are calculated here.

In [ ]:
if manifest is not None:
    from collections import Counter
    import pandas as pd

    class_to_idx = manifest["class_to_idx"]
    split_rows = []
    duplicate_sets = {}
    signer_sets = {}
    for split_name, rows in manifest["splits"].items():
        counts = Counter(row["label"] for row in rows)
        split_rows.extend({"split": split_name, "class": label, "images": counts.get(label, 0)} for label in class_to_idx)
        duplicate_sets[split_name] = {row["duplicate_group"] for row in rows}
        signer_sets[split_name] = {row.get("signer_id") for row in rows if row.get("signer_id")}
        assert {row["label_idx"] for row in rows} == set(class_to_idx.values()), f"{split_name} is missing one or more classes"

    for first, second in (("train", "validation"), ("train", "test"), ("validation", "test")):
        assert not duplicate_sets[first] & duplicate_sets[second], f"Duplicate leakage: {first} vs {second}"
    if manifest["split_mode"].startswith("signer-independent"):
        for first, second in (("train", "validation"), ("train", "test"), ("validation", "test")):
            assert not signer_sets[first] & signer_sets[second], f"Signer leakage: {first} vs {second}"

    split_counts = pd.DataFrame(split_rows)
    display(split_counts.pivot(index="class", columns="split", values="images").fillna(0).astype(int))
    display(split_counts.groupby("split")["images"].sum().rename("image_count").to_frame())
    print("Split coverage and duplicate-group integrity checks passed.")
    if not manifest["split_mode"].startswith("signer-independent"):
        print("LIMITATION: this is not a signer-independent split. Use an independent webcam test set.")
else:
    print("Split validation skipped until a valid dataset path is configured.")

## RGB preprocessing and train-only augmentation check

Both pipelines resize to 224 × 224, convert images to RGB, convert to tensors, and normalize with ImageNet statistics. The training pipeline additionally applies small affine and color perturbations; validation and test preprocessing is deterministic. No horizontal flip is used because it may change sign meaning.

In [ ]:
if manifest is not None:
    from PIL import Image
    import matplotlib.pyplot as plt
    import torch

    image_size = int(config["data"].get("image_size", 224))
    validation_transform = build_transforms(image_size=image_size, train=False)
    train_transform = build_transforms(image_size=image_size, train=True)
    sample = manifest["splits"]["train"][0]
    with Image.open(sample["path"]) as image:
        rgb = image.convert("RGB")
        validation_tensor = validation_transform(rgb)
        augmented_tensor = train_transform(rgb)
    assert validation_tensor.shape == (3, image_size, image_size)
    assert augmented_tensor.shape == validation_tensor.shape
    print(f"Input mode converted to RGB; output tensor shape: {tuple(validation_tensor.shape)}")
    print(f"Normalized deterministic tensor range: {validation_tensor.min().item():.2f} .. {validation_tensor.max().item():.2f}")
    figure, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(rgb)
    axes[0].set_title("Original RGB sample")
    axes[1].imshow(augmented_tensor.permute(1, 2, 0).mul(torch.tensor([0.229, 0.224, 0.225])).add(torch.tensor([0.485, 0.456, 0.406])).clamp(0, 1))
    axes[1].set_title("Example training augmentation")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Transform validation skipped until a dataset is available.")

### Outputs

On a configured dataset, this notebook writes the manifest and image metadata under the configured `paths.splits` and `paths.processed_data` folders. Raw images are not copied, modified, or deleted.